In [ ]:
import pandas as pd
import os
from pathlib import Path

# Ruta ancla: portátil entre sistemas operativos.
# Este script debe vivir en gas-project/scripts/ (un nivel bajo la raíz del proyecto).
BASE_DIR = Path.cwd().parent
RAW_PATH = BASE_DIR / "data" / "raw" / "Contratos 2021-2025.csv"
OUTPUT_PATH = BASE_DIR / "data" / "processed" / "contratos_gas.parquet"

print("Leyendo CSV original...")
columnas = [
    'fecha_dia', 'cantidad', 'precio', 'sector_consumo',
    'modalidad', 'mercado', 'fecha_negociacion', 'nombre_vendedor',
    'nombre_comprador', 'fecha_inicial', 'fecha_final',
    'no_operacion', 'tipo_demanda'
]

df = pd.read_csv(RAW_PATH,
                  encoding='latin-1',
                  usecols=columnas,
                  parse_dates=['fecha_dia', 'fecha_inicial', 'fecha_final'])

# Normalizar nombres de empresas con variantes/errores conocidos
normalizar_empresas = {
    'EMGESA S.A ESP': 'EMGESA S.A. E.S.P.',
    'Gases del Cusiana SA Empresa de Servicios Públicos': 'GASES DEL CUSIANA S.A E.S.P',
    'Gases del Llano S.A. Empresa de Servicios Públicos': 'Gases del Llano S.A. E.S.P.',
    'KRONOS ENERGY S.A. E.S.P': 'KRONOS ENERGY S.A. E.S.P.',
    'PRIME TERMOVALLE S.A.S. E.S.P.': 'ENFRAGEN TERMOVALLE S.A.S. E.S.P.',
    'Prime Termoflores S.A.S. E.S.P.': 'Enfragen Termoflores S.A.S. E.S.P.',
    'Gases del Cusiana S.A.S Empresa de Servicios Públicos BIC': 'GASES DEL CUSIANA S.A E.S.P',
    'Gases del Llano S.A. Empresa de Servicios Públicos BIC': 'Gases del Llano S.A. E.S.P.',
    'KRONOS ENERGY S.A.S E.S.P': 'KRONOS ENERGY S.A. E.S.P.',
    'DEL LLANO S A': 'Gases del Llano S.A. E.S.P.',
}
df['nombre_comprador'] = df['nombre_comprador'].astype(str).replace(normalizar_empresas)
df['nombre_vendedor'] = df['nombre_vendedor'].astype(str).replace(normalizar_empresas)

for col in ['sector_consumo', 'modalidad', 'mercado',
            'nombre_vendedor', 'nombre_comprador', 'tipo_demanda']:
    df[col] = df[col].astype('category')

# Reparar encoding en las categorías (mojibake de latin-1 a utf-8)
for col in ['sector_consumo', 'modalidad', 'mercado',
            'nombre_vendedor', 'nombre_comprador', 'tipo_demanda']:
    df[col] = df[col].cat.rename_categories(
        lambda x: x.encode('latin-1').decode('utf-8') if isinstance(x, str) else x
    )

df.to_parquet(OUTPUT_PATH,
              engine='pyarrow', index=False,
              version='2.6',
              coerce_timestamps='ms',
              allow_truncated_timestamps=True,
              compression='brotli')

size = os.path.getsize(OUTPUT_PATH) / 1024**2
print(f"✅ Contratos: {size:.1f} MB | {len(df):,} filas")
print(f"   Rango fecha_dia: {df['fecha_dia'].min().date()} — {df['fecha_dia'].max().date()}")

NameError: name '__file__' is not defined